# Catch logic errors in text-to-SQL

Score five fixed translations of shopper analytics questions. An alias change may be harmless; filtering out pending orders changes the total. You will compare two hosted judges with local string checks and actual query results. Use Python 3.12 and Future AGI credentials. The ten hosted judgments can consume credits.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7"], check=True)

In [ ]:
from getpass import getpass
import os

for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")
    if not os.environ[name].strip():
        raise ValueError(f"{name} is required")

## Create the disposable SQLite fixture

In [ ]:
import csv
import os
import sqlite3
from fi.evals import Evaluator, evaluate

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.executescript("""
    CREATE TABLE customers (
        id    INTEGER PRIMARY KEY,
        name  TEXT NOT NULL,
        email TEXT NOT NULL,
        city  TEXT
    );
    CREATE TABLE orders (
        id          INTEGER PRIMARY KEY,
        customer_id INTEGER REFERENCES customers(id),
        amount      REAL NOT NULL,
        status      TEXT NOT NULL,
        created_at  TEXT NOT NULL
    );

    INSERT INTO customers VALUES (1, 'Alice Johnson', 'alice@example.com', 'New York');
    INSERT INTO customers VALUES (2, 'Bob Smith',     'bob@example.com',   'Austin');
    INSERT INTO customers VALUES (3, 'Carol White',   'carol@example.com', 'Chicago');

    INSERT INTO orders VALUES (1, 1, 120.00, 'completed', '2024-01-10');
    INSERT INTO orders VALUES (2, 1,  80.50, 'completed', '2024-02-15');
    INSERT INTO orders VALUES (3, 2, 200.00, 'pending',   '2024-03-01');
    INSERT INTO orders VALUES (4, 3,  55.25, 'completed', '2024-03-10');
    INSERT INTO orders VALUES (5, 2, 175.00, 'cancelled', '2024-03-20');
""")


conn.execute("PRAGMA query_only = ON")


def run_sql(sql: str) -> list:
    """Compare these fixed SELECT examples as unordered rows; preserve duplicates."""
    return sorted(cursor.execute(sql).fetchall())


test_cases = [
    {
        "question": "Get all customer names",
        "expected_sql": "SELECT name FROM customers;",
        "generated_sql": "SELECT name FROM customers;",
    },
    {
        "question": "Find completed orders",
        "expected_sql": "SELECT * FROM orders WHERE status = 'completed';",
        "generated_sql": "SELECT * FROM orders WHERE status='completed';",
    },
    {
        "question": "Total order amount per customer",
        "expected_sql": "SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id;",
        "generated_sql": "SELECT customer_id, SUM(amount) FROM orders GROUP BY customer_id;",
    },
    {
        "question": "Customers who placed completed orders",
        "expected_sql": "SELECT name FROM customers WHERE id IN (SELECT customer_id FROM orders WHERE status = 'completed');",
        "generated_sql": "SELECT DISTINCT c.name FROM customers c JOIN orders o ON c.id = o.customer_id WHERE o.status = 'completed';",
    },
    {
        "question": "Total amount across all orders",
        "expected_sql": "SELECT SUM(amount) FROM orders;",
        "generated_sql": "SELECT SUM(amount) FROM orders WHERE status = 'completed';",
    },
]

print(f"{len(test_cases)} test cases loaded, database ready.")

## Judge the question and candidate SQL

In [ ]:
print(f"{'Question':<40}  text_to_sql")
print("-" * 55)

for tc in test_cases:
    result = evaluator.evaluate(
        eval_templates="text_to_sql",
        inputs={
            "input": tc["question"],
            "output": tc["generated_sql"],
        },
        model_name="turing_small",
    )
    eval_result = result.eval_results[0]
    if eval_result.output not in ("Passed", "Failed"):
        raise RuntimeError(f"No verdict for {tc['question']}: {eval_result.reason}")
    tc["intent"] = eval_result
    print(f"{tc['question']:<40}  {eval_result.output}")

## Compare the candidate with a reference query

In [ ]:
print(f"{'Question':<40}  ground_truth_match")
print("-" * 62)

for tc in test_cases:
    result = evaluator.evaluate(
        eval_templates="ground_truth_match",
        inputs={
            "generated_value": tc["generated_sql"],
            "expected_value": tc["expected_sql"],
        },
        model_name="turing_small",
    )
    eval_result = result.eval_results[0]
    if eval_result.output not in ("Passed", "Failed"):
        raise RuntimeError(f"No verdict for {tc['question']}: {eval_result.reason}")
    tc["reference"] = eval_result
    print(f"{tc['question']:<40}  {eval_result.output}")

## Check exact text and edit similarity locally

In [ ]:
SIMILARITY_THRESHOLD = 0.85  # flag dissimilar text for review, not automatic rejection

print(f"{'Question':<40}  {'Exact':>6}  {'Similarity':>11}")
print("-" * 62)

for tc in test_cases:
    exact = evaluate(
        "equals",
        output=tc["generated_sql"],
        expected_output=tc["expected_sql"],
    )
    sim = evaluate(
        "levenshtein_similarity",
        output=tc["generated_sql"],
        expected_output=tc["expected_sql"],
    )
    tc["exact"], tc["similarity"] = exact.passed, sim.score
    exact_str = "PASS" if exact.passed else "FAIL"
    sim_str = f"{sim.score:.2f}" + ("  low" if sim.score < SIMILARITY_THRESHOLD else "")
    print(f"{tc['question']:<40}  {exact_str:>6}  {sim_str:>11}")

## Compare returned rows on the fixture

In [ ]:
print(f"{'Question':<40}  Execution Match")
print("-" * 60)

for tc in test_cases:
    gen_rows = run_sql(tc["generated_sql"])
    ref_rows = run_sql(tc["expected_sql"])
    match = gen_rows == ref_rows
    tc["exec_match"] = match
    status = "PASS" if match else "FAIL"
    print(f"{tc['question']:<40}  {status}")
    if not match:
        print(f"  Generated: {gen_rows}")
        print(f"  Reference: {ref_rows}")

## Bring the four signals together

In [ ]:
print(f"{'Question':<40}  {'SQL':>6}  {'GT':>6}  {'Exact':>6}  {'Sim':>5}  {'Exec':>5}")
print("-" * 80)

with open("sql-eval-ids.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["question", "intent_id", "intent_verdict", "reference_id", "reference_verdict"])
    for tc in test_cases:
        intent, reference = tc["intent"], tc["reference"]
        writer.writerow([tc["question"], intent.eval_id, intent.output, reference.eval_id, reference.output])
        print(f"{tc['question']:<40}  {intent.output:>6}  {reference.output:>6}  "
              f"{'PASS' if tc['exact'] else 'FAIL':>6}  {tc['similarity']:>5.2f}  "
              f"{'PASS' if tc['exec_match'] else 'FAIL':>5}")

## Repair the extra filter

In [ ]:
test_cases[4]["generated_sql"] = "SELECT SUM(amount) FROM orders;"

tc = test_cases[4]
gen_rows = run_sql(tc["generated_sql"])
ref_rows = run_sql(tc["expected_sql"])
exec_pass = gen_rows == ref_rows

print(f"{tc['question']:<40}  Execution Match")
print(f"{tc['question']:<40}  {'PASS' if exec_pass else 'FAIL'}  {gen_rows}")

## Check your results

The two SQL statements in the last case return different totals before the fix. Read each saved judgment by ID rather than running the hosted judges again.

In [ ]:
assert [tc["exec_match"] for tc in test_cases] == [True, True, True, True, False]
assert exec_pass and gen_rows == [(630.75,)]
for tc in test_cases:
    for label, template in [("intent", "text_to_sql"), ("reference", "ground_truth_match")]:
        judgment = tc[label]
        stored = evaluator.get_eval_result(judgment.eval_id)["result"]["result"]
        assert stored["name"] == template and stored["value"] == judgment.output
print("Retrieved 10 judgments; the corrected query includes every order.")
conn.close()

A matching result on this fixture does not establish that two queries are equivalent on every dataset. These fixed SELECT statements use an in-memory database, not a production connection.